In [1]:
#Starts from here
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler 
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
from tqdm import tqdm

def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    # Identify columns with variance below the threshold
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [2]:
#Combined features and then selection

df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Descriptors/Train_2d_3d_all_descriptors.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_desc_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Descriptors/Test_2d_3d_all_descriptors.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test =df_desc_test.dropna()
df_desc_test =  df_desc_test[df_desc_train.columns]


# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Fingerprints/Train/All_fingerprints_train.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_fp_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Fingerprints/Test/All_fingerprints_test.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test =  df_fp_test[df_fp_train.columns]


#Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_emb_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test =  df_emb_test[df_emb_train.columns]

#ATomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Atomic/Train_all_atomic_desc.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_atomic_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
# df_atomic_train =pd.concat( [df_train['SMILES'], df_train.select_dtypes(include=['number'])], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Atomic/Test_all_atomic_desc.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test =  df_atomic_test[df_atomic_train.columns]


print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]

df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]

df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]

print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)


/tmp/ipykernel_369462/3312508411.py:3: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Descriptors/Train_2d_3d_all_descriptors.csv')
/tmp/ipykernel_369462/3312508411.py:11: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/PAMPA/features/Descriptors/Test_2d_3d_all_descriptors.csv')


XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(5568, 272)
(1392, 272)
(5568, 1125)
(1392, 1125)
(5568, 763)
(1392, 763)
(5568, 12)
(1392, 12)


In [3]:
merge_keys = ['ID', 'SMILES', 'Permeability']

merged_train = df_desc_train.merge(df_fp_train, on=merge_keys)
merged_train = merged_train.merge(df_emb_train, on=merge_keys)
merged_train = merged_train.merge(df_atomic_train, on=merge_keys)

merged_test = df_desc_test.merge(df_fp_test, on=merge_keys)
merged_test = merged_test.merge(df_emb_test, on=merge_keys)
merged_test = merged_test.merge(df_atomic_test, on=merge_keys)

In [4]:
X_train = merged_train.drop(columns=['ID', 'SMILES']).select_dtypes(include=['number'])
selected_final_features = features(X_train, target_column='Permeability')

train = pd.concat([merged_train[['ID', 'SMILES', 'Permeability']], X_train[selected_final_features]], axis=1)
test = merged_test[train.columns] 

print('selected_final_features', selected_final_features )
print("Final Train shape:", train.shape)
print("Final Test shape:", test.shape)

selected_final_features ['MinEStateIndex', 'qed', 'SPS', 'FpDensityMorgan1', 'BCUT2D_MWHI', 'AvgIpc', 'BalabanJ_x', 'Ipc', 'EState_VSA11', 'VSA_EState10', 'fr_Ar_N', 'fr_alkyl_halide', 'fr_allylic_oxid', 'fr_aryl_methyl', 'fr_methoxy', 'fr_para_hydroxylation', 'fr_piperdine', 'fr_priamide', 'AdjacencyMatrix.6', 'AdjacencyMatrix.9', 'AdjacencyMatrix.11', 'AATS', 'AATS.3', 'AATS.4', 'AATS.10', 'AATS.18', 'AATS.23', 'AATS.24', 'AATS.26', 'AATS.27', 'AATS.28', 'AATS.32', 'AATS.33', 'AATS.34', 'AATS.40', 'AATS.48', 'AATS.96', 'AATS.97', 'ATSC.2', 'ATSC.5', 'ATSC.7', 'ATSC.8', 'ATSC.12', 'ATSC.13', 'ATSC.14', 'ATSC.15', 'ATSC.16', 'ATSC.17', 'ATSC.20', 'ATSC.21', 'ATSC.22', 'ATSC.23', 'ATSC.24', 'ATSC.25', 'ATSC.26', 'ATSC.29', 'ATSC.30', 'ATSC.32', 'ATSC.37', 'ATSC.44', 'ATSC.50', 'ATSC.73', 'ATSC.76', 'ATSC.80', 'ATSC.84', 'ATSC.92', 'ATSC.105', 'ATSC.106', 'AATSC.9', 'AATSC.11', 'AATSC.15', 'AATSC.16', 'AATSC.36', 'AATSC.43', 'AATSC.57', 'AATSC.60', 'GATS.1', 'GATS.4', 'GATS.5', 'GATS.12'

In [5]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.9)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.9)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [6]:
X_train = train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test[X_train.columns]
y_test = test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 2141)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 2141)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 21.557835 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 254463
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2107
[LightGBM] [Info] Start training from score -5.749858
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 25.463228 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 254476
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2109
[LightGBM] [Info] Start training from score -5.749513
[LightGBM] [Info] Auto-choosing col-wise multi-thr

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1320,0.2630,0.3633,0.7880,0.8877,0.8681,0.2058,0.3241,0.4536,0.6764,0.8227,0.8041
DecisionTreeRegressor,0.2824,0.3850,0.5314,0.5465,0.7793,0.7550,0.2377,0.3482,0.4876,0.6261,0.7954,0.7723
RandomForestRegressor,0.1344,0.2668,0.3666,0.7842,0.8856,0.8652,0.2066,0.3233,0.4546,0.6750,0.8217,0.8033
GradientBoostingRegressor,0.1321,0.2646,0.3634,0.7879,0.8876,0.8666,0.2087,0.3278,0.4569,0.6717,0.8199,0.7988
AdaBoostRegressor,0.1655,0.3177,0.4069,0.7341,0.8654,0.8425,0.2321,0.3677,0.4817,0.6350,0.8032,0.7831
XGBRegressor,0.1503,0.2828,0.3877,0.7585,0.8723,0.8517,0.2046,0.3213,0.4524,0.6781,0.8242,0.8079
ExtraTreesRegressor,0.1358,0.2676,0.3685,0.7819,0.8843,0.8652,0.2055,0.3208,0.4534,0.6767,0.8228,0.8062
LinearRegression,0.6649,0.4650,0.8154,-0.0680,0.6023,0.7230,0.4279,0.4247,0.6541,0.3270,0.6660,0.7017
KNeighborsRegressor,0.1974,0.3254,0.4443,0.6829,0.8300,0.8023,0.2224,0.3366,0.4716,0.6502,0.8096,0.7939
SVR,0.1558,0.2832,0.3948,0.7497,0.8664,0.8515,0.2148,0.3291,0.4635,0.6622,0.8144,0.8060


In [7]:
result_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/combined_features_pampa.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/prediction_data_combined_features_pampa.csv')

In [8]:
X = train.drop(columns=['ID', 'SMILES', 'Permeability'])
y = train['Permeability']

rf = RandomForestRegressor(n_estimators=100, random_state=101, n_jobs=-1)
rf.fit(X, y)

importances = rf.feature_importances_
feature_names = X.columns


In [9]:
#Top 10 features
n = 10  
top_10_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_10_features = feature_names[top_10_indices].tolist()  # convert to list

# Output the list
print("Top", 10, "features:\n")
print(top_10_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_10_features]], axis=1)
test_df = test[train.columns] 

Top 10 features:

['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338', 'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427', 'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687', 'x_fine_emb_MFXL571']


In [10]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
print('Features: ', X_train.columns)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 10)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 10)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Features:  Index(['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338',
       'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427',
       'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687',
       'x_fine_emb_MFXL571'],
      dtype='object')
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073656 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 10
[LightGBM] [Info] Start training from score -5.749858
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.097085 seconds.
You can set `force_c

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1382,0.2728,0.3717,0.7781,0.8821,0.8589,0.2169,0.3351,0.4657,0.6589,0.8121,0.7900
DecisionTreeRegressor,0.2851,0.3913,0.5340,0.5420,0.7729,0.7477,0.2539,0.3577,0.5038,0.6007,0.7792,0.7597
RandomForestRegressor,0.1402,0.2751,0.3744,0.7749,0.8804,0.8578,0.2167,0.3334,0.4655,0.6591,0.8122,0.7899
GradientBoostingRegressor,0.1375,0.2733,0.3708,0.7792,0.8827,0.8601,0.2170,0.3368,0.4659,0.6586,0.8120,0.7887
AdaBoostRegressor,0.1938,0.3408,0.4403,0.6887,0.8429,0.8337,0.2581,0.3888,0.5080,0.5940,0.7811,0.7743
XGBRegressor,0.1534,0.2896,0.3917,0.7535,0.8690,0.8432,0.2227,0.3380,0.4719,0.6498,0.8071,0.7832
ExtraTreesRegressor,0.1406,0.2751,0.3750,0.7741,0.8801,0.8582,0.2169,0.3327,0.4658,0.6588,0.8121,0.7935
LinearRegression,0.1369,0.2762,0.3701,0.7800,0.8832,0.8594,0.2196,0.3393,0.4686,0.6546,0.8094,0.7862
KNeighborsRegressor,0.1697,0.3038,0.4120,0.7274,0.8555,0.8313,0.2308,0.3434,0.4804,0.6370,0.8012,0.7792
SVR,0.1357,0.2705,0.3684,0.7820,0.8846,0.8617,0.2209,0.3352,0.4700,0.6526,0.8093,0.7891


In [11]:
result_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/combined_top_10_features_pampa.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/prediction_data_combined_top_10_features_pampa.csv')

In [12]:
#Top 20 features
n = 20  
top_20_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_20_features = feature_names[top_20_indices].tolist()  # convert to list

# Output the list
print("Top", 20, "features:\n")
print(top_20_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_20_features]], axis=1)
test_df = test[train.columns] 

Top 20 features:

['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338', 'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427', 'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687', 'x_fine_emb_MFXL571', 'x_fine_emb_MFXL681', 'x_fine_emb_MFXL452', 'x_fine_emb_MFXL90', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL538', 'x_fine_emb_MFXL335', 'x_fine_emb_MFXL640', 'x_fine_emb_MFXL207', 'x_fine_emb_MFXL271', 'x_fine_emb_MFXL658']


In [13]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
print('Features: ', X_train.columns)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 20)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 20)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Features:  Index(['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338',
       'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427',
       'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687',
       'x_fine_emb_MFXL571', 'x_fine_emb_MFXL681', 'x_fine_emb_MFXL452',
       'x_fine_emb_MFXL90', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL538',
       'x_fine_emb_MFXL335', 'x_fine_emb_MFXL640', 'x_fine_emb_MFXL207',
       'x_fine_emb_MFXL271', 'x_fine_emb_MFXL658'],
      dtype='object')
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.228162 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5100
[LightGBM] [Info] Number of da

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1345,0.2689,0.3667,0.7840,0.8854,0.8633,0.2141,0.3324,0.4627,0.6633,0.8149,0.7946
DecisionTreeRegressor,0.2637,0.3769,0.5135,0.5764,0.7878,0.7594,0.2351,0.3480,0.4849,0.6302,0.7979,0.7788
RandomForestRegressor,0.1363,0.2707,0.3692,0.7811,0.8839,0.8625,0.2118,0.3281,0.4602,0.6669,0.8169,0.7966
GradientBoostingRegressor,0.1350,0.2704,0.3674,0.7832,0.8850,0.8625,0.2161,0.3354,0.4648,0.6602,0.8131,0.7898
AdaBoostRegressor,0.1766,0.3256,0.4202,0.7164,0.8534,0.8293,0.2433,0.3731,0.4932,0.6174,0.7906,0.7716
XGBRegressor,0.1551,0.2886,0.3939,0.7508,0.8680,0.8439,0.2162,0.3322,0.4650,0.6599,0.8132,0.7907
ExtraTreesRegressor,0.1379,0.2711,0.3714,0.7784,0.8825,0.8622,0.2135,0.3292,0.4620,0.6642,0.8155,0.7982
LinearRegression,0.1350,0.2736,0.3674,0.7832,0.8850,0.8612,0.2213,0.3406,0.4705,0.6519,0.8078,0.7847
KNeighborsRegressor,0.1645,0.2978,0.4056,0.7358,0.8603,0.8347,0.2256,0.3408,0.4750,0.6451,0.8063,0.7870
SVR,0.1337,0.2656,0.3656,0.7853,0.8865,0.8646,0.2165,0.3316,0.4653,0.6595,0.8135,0.7937


In [14]:
result_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/combined_top_20_features_pampa.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/prediction_data_combined_top_20_features_pampa.csv')

In [15]:
#Top 50 features
n = 50  
top_50_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_50_features = feature_names[top_50_indices].tolist()  # convert to list

# Output the list
print("Top", 50, "features:\n")
print(top_50_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_50_features]], axis=1)
test_df = test[train.columns] 

Top 50 features:

['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338', 'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427', 'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687', 'x_fine_emb_MFXL571', 'x_fine_emb_MFXL681', 'x_fine_emb_MFXL452', 'x_fine_emb_MFXL90', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL538', 'x_fine_emb_MFXL335', 'x_fine_emb_MFXL640', 'x_fine_emb_MFXL207', 'x_fine_emb_MFXL271', 'x_fine_emb_MFXL658', 'x_fine_emb_MFXL486', 'x_fine_emb_MFXL315', 'x_fine_emb_MFXL27', 'x_fine_emb_MFXL259', 'x_fine_emb_MFXL140', 'x_fine_emb_MFXL178', 'x_fine_emb_MFXL192', 'MOMI-XY', 'x_fine_emb_MFXL25', 'x_fine_emb_MFXL136', 'x_fine_emb_MFXL50', 'x_fine_emb_MFXL230', 'x_fine_emb_MFXL651', 'x_fine_emb_MFXL57', 'x_fine_emb_MFXL244', 'x_fine_emb_MFXL472', 'x_fine_emb_MFXL270', 'CrippenLogP', 'x_fine_emb_MFXL106', 'x_fine_emb_MFXL119', 'x_fine_emb_MFXL65', 'x_fine_emb_MFXL615', 'x_fine_emb_MFXL38', 'x_fine_emb_MFXL283', 'x_fine_emb_MFXL141', 'x_fine_emb_MFXL

In [16]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
print('Features: ', X_train.columns)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 50)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 50)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Features:  Index(['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338',
       'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427',
       'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687',
       'x_fine_emb_MFXL571', 'x_fine_emb_MFXL681', 'x_fine_emb_MFXL452',
       'x_fine_emb_MFXL90', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL538',
       'x_fine_emb_MFXL335', 'x_fine_emb_MFXL640', 'x_fine_emb_MFXL207',
       'x_fine_emb_MFXL271', 'x_fine_emb_MFXL658', 'x_fine_emb_MFXL486',
       'x_fine_emb_MFXL315', 'x_fine_emb_MFXL27', 'x_fine_emb_MFXL259',
       'x_fine_emb_MFXL140', 'x_fine_emb_MFXL178', 'x_fine_emb_MFXL192',
       'MOMI-XY', 'x_fine_emb_MFXL25', 'x_fine_emb_MFXL136',
       'x_fine_emb_MF

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.6473060745476737


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1314,0.2649,0.3624,0.7890,0.8883,0.8666,0.2102,0.3285,0.4584,0.6695,0.8185,0.7981
DecisionTreeRegressor,0.2757,0.3823,0.5251,0.5571,0.7823,0.7563,0.2417,0.3499,0.4916,0.6198,0.7915,0.7710
RandomForestRegressor,0.1319,0.2659,0.3631,0.7882,0.8879,0.8658,0.2090,0.3256,0.4572,0.6713,0.8196,0.8013
GradientBoostingRegressor,0.1323,0.2665,0.3638,0.7875,0.8874,0.8640,0.2135,0.3321,0.4621,0.6642,0.8155,0.7939
AdaBoostRegressor,0.1691,0.3178,0.4112,0.7284,0.8599,0.8355,0.2395,0.3688,0.4894,0.6233,0.7938,0.7725
XGBRegressor,0.1524,0.2845,0.3904,0.7552,0.8702,0.8471,0.2070,0.3251,0.4550,0.6743,0.8218,0.8022
ExtraTreesRegressor,0.1343,0.2671,0.3664,0.7844,0.8857,0.8647,0.2089,0.3248,0.4570,0.6714,0.8197,0.8023
LinearRegression,0.1312,0.2693,0.3623,0.7892,0.8884,0.8637,0.2160,0.3349,0.4647,0.6603,0.8129,0.7889
KNeighborsRegressor,0.1673,0.3001,0.4090,0.7314,0.8578,0.8297,0.2227,0.3373,0.4719,0.6497,0.8087,0.7887
SVR,0.1321,0.2634,0.3635,0.7878,0.8879,0.8659,0.2134,0.3267,0.4619,0.6644,0.8162,0.7981


In [17]:
result_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/combined_top_50_features_pampa.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/prediction_data_combined_top_50_features_pampa.csv')

In [18]:
#Top 100 features
n = 100  
top_100_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_100_features = feature_names[top_100_indices].tolist()  # convert to list

# Output the list
print("Top", 100, "features:\n")
print(top_100_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_100_features]], axis=1)
test_df = test[train.columns] 

Top 100 features:

['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338', 'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427', 'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687', 'x_fine_emb_MFXL571', 'x_fine_emb_MFXL681', 'x_fine_emb_MFXL452', 'x_fine_emb_MFXL90', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL538', 'x_fine_emb_MFXL335', 'x_fine_emb_MFXL640', 'x_fine_emb_MFXL207', 'x_fine_emb_MFXL271', 'x_fine_emb_MFXL658', 'x_fine_emb_MFXL486', 'x_fine_emb_MFXL315', 'x_fine_emb_MFXL27', 'x_fine_emb_MFXL259', 'x_fine_emb_MFXL140', 'x_fine_emb_MFXL178', 'x_fine_emb_MFXL192', 'MOMI-XY', 'x_fine_emb_MFXL25', 'x_fine_emb_MFXL136', 'x_fine_emb_MFXL50', 'x_fine_emb_MFXL230', 'x_fine_emb_MFXL651', 'x_fine_emb_MFXL57', 'x_fine_emb_MFXL244', 'x_fine_emb_MFXL472', 'x_fine_emb_MFXL270', 'CrippenLogP', 'x_fine_emb_MFXL106', 'x_fine_emb_MFXL119', 'x_fine_emb_MFXL65', 'x_fine_emb_MFXL615', 'x_fine_emb_MFXL38', 'x_fine_emb_MFXL283', 'x_fine_emb_MFXL141', 'x_fine_emb_MFX

In [19]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
print('Features: ', X_train.columns)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 100)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 100)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Features:  Index(['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338',
       'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427',
       'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687',
       'x_fine_emb_MFXL571', 'x_fine_emb_MFXL681', 'x_fine_emb_MFXL452',
       'x_fine_emb_MFXL90', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL538',
       'x_fine_emb_MFXL335', 'x_fine_emb_MFXL640', 'x_fine_emb_MFXL207',
       'x_fine_emb_MFXL271', 'x_fine_emb_MFXL658', 'x_fine_emb_MFXL486',
       'x_fine_emb_MFXL315', 'x_fine_emb_MFXL27', 'x_fine_emb_MFXL259',
       'x_fine_emb_MFXL140', 'x_fine_emb_MFXL178', 'x_fine_emb_MFXL192',
       'MOMI-XY', 'x_fine_emb_MFXL25', 'x_fine_emb_MFXL136',
       'x_fine_emb_

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.5980503368840411


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1311,0.2642,0.3621,0.7894,0.8885,0.8676,0.2099,0.3281,0.4581,0.6699,0.8188,0.7989
DecisionTreeRegressor,0.2605,0.3731,0.5104,0.5815,0.7932,0.7611,0.2402,0.3533,0.4901,0.6222,0.7932,0.7761
RandomForestRegressor,0.1322,0.2654,0.3635,0.7877,0.8876,0.8662,0.2085,0.3256,0.4567,0.6720,0.8199,0.8007
GradientBoostingRegressor,0.1329,0.2660,0.3646,0.7865,0.8869,0.8650,0.2139,0.3316,0.4625,0.6636,0.8151,0.7937
AdaBoostRegressor,0.1684,0.3184,0.4103,0.7296,0.8629,0.8423,0.2366,0.3683,0.4864,0.6279,0.7985,0.7800
XGBRegressor,0.1491,0.2822,0.3862,0.7605,0.8730,0.8526,0.2123,0.3303,0.4608,0.6661,0.8167,0.7990
ExtraTreesRegressor,0.1325,0.2653,0.3640,0.7872,0.8873,0.8661,0.2082,0.3235,0.4563,0.6725,0.8203,0.8036
LinearRegression,0.1299,0.2671,0.3604,0.7913,0.8896,0.8654,0.2139,0.3336,0.4625,0.6636,0.8149,0.7932
KNeighborsRegressor,0.1702,0.3021,0.4126,0.7266,0.8547,0.8244,0.2265,0.3424,0.4759,0.6437,0.8048,0.7859
SVR,0.1326,0.2646,0.3641,0.7870,0.8873,0.8666,0.2133,0.3301,0.4618,0.6645,0.8159,0.8004


In [20]:
result_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/combined_top_100_features_pampa.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/prediction_data_combined_top_100_features_pampa.csv')

In [21]:
#Top 200 features
n = 200  
top_200_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_200_features = feature_names[top_200_indices].tolist()  # convert to list

# Output the list
print("Top", 200, "features:\n")
print(top_200_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_200_features]], axis=1)
test_df = test[train.columns]

Top 200 features:

['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338', 'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427', 'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687', 'x_fine_emb_MFXL571', 'x_fine_emb_MFXL681', 'x_fine_emb_MFXL452', 'x_fine_emb_MFXL90', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL538', 'x_fine_emb_MFXL335', 'x_fine_emb_MFXL640', 'x_fine_emb_MFXL207', 'x_fine_emb_MFXL271', 'x_fine_emb_MFXL658', 'x_fine_emb_MFXL486', 'x_fine_emb_MFXL315', 'x_fine_emb_MFXL27', 'x_fine_emb_MFXL259', 'x_fine_emb_MFXL140', 'x_fine_emb_MFXL178', 'x_fine_emb_MFXL192', 'MOMI-XY', 'x_fine_emb_MFXL25', 'x_fine_emb_MFXL136', 'x_fine_emb_MFXL50', 'x_fine_emb_MFXL230', 'x_fine_emb_MFXL651', 'x_fine_emb_MFXL57', 'x_fine_emb_MFXL244', 'x_fine_emb_MFXL472', 'x_fine_emb_MFXL270', 'CrippenLogP', 'x_fine_emb_MFXL106', 'x_fine_emb_MFXL119', 'x_fine_emb_MFXL65', 'x_fine_emb_MFXL615', 'x_fine_emb_MFXL38', 'x_fine_emb_MFXL283', 'x_fine_emb_MFXL141', 'x_fine_emb_MFX

In [22]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
print('Features: ', X_train.columns)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 200)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 200)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Features:  Index(['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338',
       'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427',
       'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687',
       'x_fine_emb_MFXL571',
       ...
       'x_fine_emb_MFXL251', 'x_fine_emb_MFXL733', 'TDB10m',
       'x_fine_emb_MFXL468', 'x_fine_emb_MFXL573', 'x_fine_emb_MFXL732',
       'x_fine_emb_MFXL108', 'x_fine_emb_MFXL704', 'x_fine_emb_MFXL17',
       'x_fine_emb_MFXL198'],
      dtype='object', length=200)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.186657 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1309,0.2635,0.3619,0.7897,0.8887,0.8691,0.2087,0.3271,0.4569,0.6717,0.8199,0.8012
DecisionTreeRegressor,0.2635,0.3754,0.5133,0.5768,0.7891,0.7655,0.2368,0.3484,0.4867,0.6275,0.7957,0.7723
RandomForestRegressor,0.1312,0.2642,0.3622,0.7893,0.8885,0.8675,0.2072,0.3244,0.4552,0.6741,0.8212,0.8026
GradientBoostingRegressor,0.1330,0.2666,0.3647,0.7864,0.8868,0.8655,0.2124,0.3305,0.4609,0.6659,0.8165,0.7943
AdaBoostRegressor,0.1644,0.3145,0.4054,0.7360,0.8659,0.8424,0.2359,0.3708,0.4857,0.6290,0.7986,0.7748
XGBRegressor,0.1518,0.2850,0.3896,0.7562,0.8708,0.8489,0.2100,0.3261,0.4583,0.6697,0.8190,0.8010
ExtraTreesRegressor,0.1329,0.2653,0.3645,0.7866,0.8870,0.8669,0.2058,0.3223,0.4537,0.6763,0.8226,0.8061
LinearRegression,0.1296,0.2664,0.3600,0.7918,0.8899,0.8665,0.2131,0.3325,0.4616,0.6649,0.8158,0.7953
KNeighborsRegressor,0.1756,0.3034,0.4191,0.7179,0.8503,0.8222,0.2208,0.3383,0.4698,0.6528,0.8105,0.7929
SVR,0.1350,0.2667,0.3674,0.7832,0.8852,0.8661,0.2134,0.3290,0.4620,0.6643,0.8156,0.7981


In [23]:
result_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/combined_top_200_features_pampa.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/prediction_data_combined_top_200_features_pampa.csv')

In [24]:
#Top 500 features
n = 500  
top_500_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_500_features = feature_names[top_500_indices].tolist()  # convert to list

# Output the list
print("Top", 500, "features:\n")
print(top_500_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_500_features]], axis=1)
test_df = test[train.columns]

Top 500 features:

['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338', 'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427', 'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687', 'x_fine_emb_MFXL571', 'x_fine_emb_MFXL681', 'x_fine_emb_MFXL452', 'x_fine_emb_MFXL90', 'x_fine_emb_MFXL440', 'x_fine_emb_MFXL538', 'x_fine_emb_MFXL335', 'x_fine_emb_MFXL640', 'x_fine_emb_MFXL207', 'x_fine_emb_MFXL271', 'x_fine_emb_MFXL658', 'x_fine_emb_MFXL486', 'x_fine_emb_MFXL315', 'x_fine_emb_MFXL27', 'x_fine_emb_MFXL259', 'x_fine_emb_MFXL140', 'x_fine_emb_MFXL178', 'x_fine_emb_MFXL192', 'MOMI-XY', 'x_fine_emb_MFXL25', 'x_fine_emb_MFXL136', 'x_fine_emb_MFXL50', 'x_fine_emb_MFXL230', 'x_fine_emb_MFXL651', 'x_fine_emb_MFXL57', 'x_fine_emb_MFXL244', 'x_fine_emb_MFXL472', 'x_fine_emb_MFXL270', 'CrippenLogP', 'x_fine_emb_MFXL106', 'x_fine_emb_MFXL119', 'x_fine_emb_MFXL65', 'x_fine_emb_MFXL615', 'x_fine_emb_MFXL38', 'x_fine_emb_MFXL283', 'x_fine_emb_MFXL141', 'x_fine_emb_MFX

In [25]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
print('Features: ', X_train.columns)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 500)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 500)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Features:  Index(['x_fine_emb_MFXL339', 'x_fine_emb_MFXL321', 'x_fine_emb_MFXL338',
       'x_fine_emb_MFXL190', 'x_fine_emb_MFXL691', 'x_fine_emb_MFXL427',
       'x_fine_emb_MFXL480', 'x_fine_emb_MFXL109', 'x_fine_emb_MFXL687',
       'x_fine_emb_MFXL571',
       ...
       'x_fine_emb_MFXL474', 'x_fine_emb_MFXL358', 'x_fine_emb_MFXL524',
       'x_fine_emb_MFXL598', 'x_fine_emb_MFXL307', 'x_fine_emb_MFXL547',
       'x_fine_emb_MFXL350', 'x_fine_emb_MFXL367', 'x_fine_emb_MFXL742',
       'x_fine_emb_MFXL437'],
      dtype='object', length=500)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.200678 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1316,0.2627,0.3628,0.7886,0.8881,0.8689,0.2065,0.3257,0.4544,0.6752,0.8220,0.8030
DecisionTreeRegressor,0.2685,0.3755,0.5182,0.5688,0.7884,0.7585,0.2403,0.3471,0.4902,0.6220,0.7936,0.7700
RandomForestRegressor,0.1330,0.2657,0.3647,0.7864,0.8869,0.8665,0.2065,0.3234,0.4544,0.6752,0.8218,0.8033
GradientBoostingRegressor,0.1330,0.2654,0.3647,0.7864,0.8868,0.8658,0.2100,0.3287,0.4583,0.6696,0.8187,0.7959
AdaBoostRegressor,0.1656,0.3175,0.4070,0.7340,0.8666,0.8422,0.2345,0.3705,0.4843,0.6311,0.8019,0.7790
XGBRegressor,0.1509,0.2844,0.3884,0.7577,0.8716,0.8506,0.2066,0.3227,0.4546,0.6750,0.8222,0.8050
ExtraTreesRegressor,0.1336,0.2659,0.3655,0.7854,0.8863,0.8665,0.2047,0.3209,0.4524,0.6780,0.8236,0.8068
LinearRegression,0.1351,0.2727,0.3676,0.7830,0.8852,0.8627,0.2127,0.3348,0.4612,0.6654,0.8164,0.7932
KNeighborsRegressor,0.1714,0.3010,0.4140,0.7247,0.8539,0.8278,0.2173,0.3349,0.4661,0.6582,0.8138,0.7974
SVR,0.1361,0.2665,0.3689,0.7814,0.8841,0.8666,0.2090,0.3245,0.4572,0.6712,0.8197,0.8026


In [26]:
result_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/combined_top_500_features_pampa.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/PAMPA/Results/combined_features/prediction_data_combined_top_500_features_pampa.csv')